# LeadPrior-Net Pretraining on PTB-XL

**LeadPrior-Net: Anatomical Lead-Prior Network**  
*A physiology-guided CNN-Transformer architecture for multi-territory myocardial infarction localization.*

## 1. Imports and CONFIG

In [ ]:
import ast
import csv
import json
import logging
import math
import os
import random
import re
import time
import traceback
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
    balanced_accuracy_score,
)
from torch.utils.data import DataLoader, Dataset

sns.set_theme(style='whitegrid', context='notebook')
warnings.filterwarnings('ignore', message='enable_nested_tensor is True.*')
warnings.filterwarnings('ignore', category=DeprecationWarning)

PROJECT_ROOT = Path('..').resolve() if Path.cwd().name == 'notebook' else Path('.').resolve()
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')


def env_int(name, default):
    value = os.environ.get(name, '').strip()
    if value in {'', '0'}:
        return default
    try:
        parsed = int(value)
    except ValueError:
        return default
    return parsed if parsed > 0 else default

CONFIG = {
    'seed': env_int('CLICNET_SEED', 42),
    'dataset_dir': str(PROJECT_ROOT / 'dataset' / 'ptb_xl'),
    'dataset_variant': 'beat_signals_100hz_recordwise',
    'augmentation_enabled': False,
    'balance_train_beats': True,
    'train_beats_per_class': 900,
    'output_base_dir': str(PROJECT_ROOT / 'outputs' / '4_pretrain_leadprior'),
    'run_name': RUN_TIMESTAMP,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'num_workers': 2,
    'batch_size': 64,
    'epochs': env_int('CLICNET_EPOCHS', 50),
    'learning_rate': 1e-4,
    'weight_decay': 1e-4,
    'scheduler_patience': 5,
    'early_stopping_enabled': False,
    'early_stopping_patience': 15,
    'loss_function': 'CrossEntropyLoss',
    'classification_head': 'softmax',
    'attention_prior_strength': 1.2,
    'attention_prior_learnable': True,
    'model': {
        'input_leads': 12,
        'signal_length': 65,
        'stem_channels': 32,
        'cnn_channels': 64,
        'token_dim': 128,
        'num_heads': 4,
        'cross_lead_layers': 2,
        'temporal_layers': 1,
        'transformer_ff_dim': 256,
        'dropout': 0.20,
        'temporal_segments': 20,
        'main_outputs': 4,
        'sub_outputs': 0,
    },
}

MAIN_LABELS = ['Normal', 'Anterior', 'Inferior', 'Lateral']
MAIN_LABEL_DISPLAY = ['NORM', 'AMI', 'IMI', 'Lateral-involved MI']
MAIN_LABEL_TO_INDEX = {label: idx for idx, label in enumerate(MAIN_LABELS)}
INDEX_TO_MAIN_LABEL = {idx: label for label, idx in MAIN_LABEL_TO_INDEX.items()}
LEAD_ORDER = ['I', 'aVL', 'II', 'III', 'aVF', 'aVR', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
MAIN_CLASS_LEAD_PRIORS = {
    'Normal': LEAD_ORDER,
    'Anterior': ['V1', 'V2', 'V3', 'V4'],
    'Inferior': ['II', 'III', 'aVF'],
    'Lateral': ['I', 'aVL', 'V5', 'V6', 'V1', 'V2', 'V3', 'V4', 'II', 'III', 'aVF'],
}
LATERAL_INVOLVED_PRIOR_COMPONENTS = {
    'lateral_primary': {'leads': ['I', 'aVL', 'V5', 'V6'], 'mass': 0.55},
    'inferior_overlap': {'leads': ['II', 'III', 'aVF'], 'mass': 0.20},
    'anterior_overlap': {'leads': ['V1', 'V2', 'V3', 'V4'], 'mass': 0.20},
    'posterior_proxy': {'leads': ['V1', 'V2', 'V3'], 'mass': 0.05},
}


def build_sparse_prior(active_leads, active_weight=0.90):
    n_leads = len(LEAD_ORDER)
    inactive_count = n_leads - len(active_leads)
    weights = np.full(n_leads, (1.0 - active_weight) / max(inactive_count, 1), dtype=np.float32)
    for lead in active_leads:
        weights[LEAD_ORDER.index(lead)] = active_weight / len(active_leads)
    return weights / weights.sum()


def build_lateral_involved_prior():
    weights = np.zeros(len(LEAD_ORDER), dtype=np.float32)
    for component in LATERAL_INVOLVED_PRIOR_COMPONENTS.values():
        mass = float(component['mass'])
        leads = component['leads']
        for lead in leads:
            weights[LEAD_ORDER.index(lead)] += mass / len(leads)
    return weights / weights.sum()


def build_main_class_prior_matrix(active_weight=0.90):
    matrix = []
    for class_name in MAIN_LABELS:
        if class_name == 'Normal':
            weights = np.ones(len(LEAD_ORDER), dtype=np.float32) / len(LEAD_ORDER)
        elif class_name == 'Lateral':
            weights = build_lateral_involved_prior()
        else:
            weights = build_sparse_prior(MAIN_CLASS_LEAD_PRIORS[class_name], active_weight=active_weight)
        matrix.append(weights)
    return np.stack(matrix).astype(np.float32)

MAIN_CLASS_PRIOR_MATRIX = build_main_class_prior_matrix(active_weight=0.90)

LEAD_PRIOR_CONFIG = {
    'lead_order': LEAD_ORDER,
    'main_label_order': MAIN_LABELS,
    'main_class_lead_priors': MAIN_CLASS_LEAD_PRIORS,
    'lateral_involved_definition': ['lateral', 'antero-lateral', 'infero-lateral', 'postero-lateral', 'infero-postero-lateral'],
    'lateral_involved_prior_components': LATERAL_INVOLVED_PRIOR_COMPONENTS,
    'main_class_prior_matrix': MAIN_CLASS_PRIOR_MATRIX.tolist(),
    'attention_prior_strength': CONFIG['attention_prior_strength'],
    'attention_prior_learnable': CONFIG['attention_prior_learnable'],
    'note': 'Attention is initialized with a strong class-lead prior logit bias before softmax. The fourth class is Lateral-involved MI. Its initial attention prior uses lateral leads as the primary component, with smaller anterior, inferior, and posterior-proxy components to match overlap labels in the prepared dataset.',
}
LABEL_MAPPING = {
    'model_name': 'LeadPrior-Net: Anatomical Lead-Prior Network',
    'task_type': 'single_label_4_class_softmax',
    'main_label_order': MAIN_LABELS,
    'main_label_display': MAIN_LABEL_DISPLAY,
    'class_to_index': MAIN_LABEL_TO_INDEX,
    'index_to_class': INDEX_TO_MAIN_LABEL,
    'target_format': 'integer class index',
    'head_activation_for_inference': 'softmax',
    'recommended_loss': 'CrossEntropyLoss',
}
LOSS_ROLE_MAP = {
    'CrossEntropyLoss': 'Directly optimizes mutually exclusive 4-class classification logits for Normal, Anterior, Inferior, and Lateral-involved MI.',
}

print('Device:', CONFIG['device'])
print('Dataset:', CONFIG['dataset_dir'])
print('Run:', CONFIG['run_name'])


## 2. Output directory and logging setup

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG['seed'])

OUTPUT_DIR = Path(CONFIG['output_base_dir']) / CONFIG['run_name']
CONFIG_DIR = OUTPUT_DIR / 'configs'
LOG_DIR = OUTPUT_DIR / 'logs'
CKPT_DIR = OUTPUT_DIR / 'checkpoints'
PLOT_DIR = OUTPUT_DIR / 'plots'
PRED_DIR = OUTPUT_DIR / 'predictions'
TRANSFER_DIR = OUTPUT_DIR / 'transfer_ready'
METRICS_DIR = OUTPUT_DIR / 'metrics'
PER_5_EPOCH_CKPT_DIR = CKPT_DIR / 'per_5fold'
for directory in [CONFIG_DIR, LOG_DIR, CKPT_DIR, PER_5_EPOCH_CKPT_DIR, PLOT_DIR, METRICS_DIR, PRED_DIR, TRANSFER_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

with open(CONFIG_DIR / 'config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)
with open(CONFIG_DIR / 'label_mapping.json', 'w') as f:
    json.dump(LABEL_MAPPING, f, indent=2)
with open(CONFIG_DIR / 'lead_prior_config.json', 'w') as f:
    json.dump(LEAD_PRIOR_CONFIG, f, indent=2)

logger = logging.getLogger('LeadPrior-Net logger')
logger.setLevel(logging.INFO)
logger.handlers.clear()
formatter = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
file_handler = logging.FileHandler(LOG_DIR / 'train.log')
file_handler.setFormatter(formatter)
ROOT_OUTPUT_LOG = PROJECT_ROOT / 'output.log'
root_file_handler = logging.FileHandler(ROOT_OUTPUT_LOG)
root_file_handler.setFormatter(formatter)
error_handler = logging.FileHandler(LOG_DIR / 'error.log')
error_handler.setLevel(logging.ERROR)
error_handler.setFormatter(formatter)
logger.addHandler(stream_handler)
logger.addHandler(file_handler)
logger.addHandler(root_file_handler)
logger.addHandler(error_handler)

RUN_STARTED_AT = datetime.now()
RUN_START_TIME = time.time()

logger.info('=' * 88)
logger.info('LeadPrior-Net logger | Algorithm: LeadPrior-Net: Anatomical Lead-Prior Network')
logger.info('Run started at: %s', RUN_STARTED_AT.strftime('%Y-%m-%d %H:%M:%S'))
logger.info('Run ID: %s', CONFIG['run_name'])
logger.info('Output directory: %s', OUTPUT_DIR)
logger.info('Root output log: %s', ROOT_OUTPUT_LOG)
logger.info('Dataset directory: %s', CONFIG['dataset_dir'])
logger.info('Device: %s', CONFIG['device'])
logger.info('Planned epochs: %d | Batch size: %d | Learning rate: %.3e | Weight decay: %.3e', CONFIG['epochs'], CONFIG['batch_size'], CONFIG['learning_rate'], CONFIG['weight_decay'])
logger.info('Early stopping enabled: %s | Loss: %s | Head activation: %s', CONFIG['early_stopping_enabled'], CONFIG['loss_function'], CONFIG['classification_head'])
logger.info('Main labels: %s', MAIN_LABELS)
logger.info('Loss role map: %s', LOSS_ROLE_MAP)
logger.info('=' * 88)

## 3. Data loading

In [ ]:
logger.info('Data loading started.')

def load_split(dataset_dir, split):
    split_dir = Path(dataset_dir) / split
    beat_signal_path = split_dir / 'x_beats.npy'
    record_signal_path = split_dir / 'x.npy'
    if beat_signal_path.exists():
        x = np.load(beat_signal_path).astype(np.float32)
    elif record_signal_path.exists():
        x = np.load(record_signal_path).astype(np.float32)
    else:
        raise FileNotFoundError(f'No signal array found in {split_dir}. Expected x_beats.npy or x.npy')

    labels_path = split_dir / 'labels.csv'
    metadata_path = split_dir / 'metadata.csv'
    if labels_path.exists():
        labels_df = pd.read_csv(labels_path)
    elif metadata_path.exists():
        labels_df = pd.read_csv(metadata_path)
    else:
        raise FileNotFoundError(f'No label/metadata CSV found in {split_dir}. Expected labels.csv or metadata.csv')

    metadata_df = pd.read_csv(metadata_path) if metadata_path.exists() else labels_df.copy()
    return x, labels_df, metadata_df

DATASET_DIR = Path(CONFIG['dataset_dir'])
x_train, labels_train, meta_train = load_split(DATASET_DIR, 'train')
x_val, labels_val, meta_val = load_split(DATASET_DIR, 'val')
x_test, labels_test, meta_test = load_split(DATASET_DIR, 'test')

lead_order_file = DATASET_DIR / 'lead_order.csv'
config_file = DATASET_DIR / 'config.json'
if lead_order_file.exists():
    lead_df = pd.read_csv(lead_order_file)
    lead_col = 'lead' if 'lead' in lead_df.columns else lead_df.columns[0]
    dataset_leads = lead_df[lead_col].tolist()
    if dataset_leads != LEAD_ORDER:
        raise ValueError(f'Lead order mismatch. Expected {LEAD_ORDER}, got {dataset_leads}')
elif config_file.exists():
    with open(config_file) as f:
        dataset_config = json.load(f)
    dataset_leads = dataset_config.get('lead_order')
    if dataset_leads and dataset_leads != LEAD_ORDER:
        raise ValueError(f'Lead order mismatch. Expected {LEAD_ORDER}, got {dataset_leads}')
else:
    logger.warning('No lead_order.csv or config.json found in dataset directory; using notebook LEAD_ORDER.')

if x_train.shape[1] != CONFIG['model']['signal_length']:
    logger.warning('Updating CONFIG model signal_length from %s to detected train length %s', CONFIG['model']['signal_length'], x_train.shape[1])
    CONFIG['model']['signal_length'] = int(x_train.shape[1])
    with open(CONFIG_DIR / 'config.json', 'w') as f:
        json.dump(CONFIG, f, indent=2)

logger.info('Data loading completed.')
logger.info('Train shape: %s | Val shape: %s | Test shape: %s', x_train.shape, x_val.shape, x_test.shape)
logger.info('Train/val/test beat samples: %d / %d / %d', len(x_train), len(x_val), len(x_test))
display(pd.read_csv(DATASET_DIR / 'split_summary.csv'))


## 4. Classification label mapping and validation


In [ ]:
logger.info('Classification label mapping started.')

MAIN_LABEL_TO_INDEX = {label: idx for idx, label in enumerate(MAIN_LABELS)}
INDEX_TO_MAIN_LABEL = {idx: label for label, idx in MAIN_LABEL_TO_INDEX.items()}


def parse_vector_column(value):
    if isinstance(value, np.ndarray):
        return value.astype(np.float32)
    if isinstance(value, list):
        return np.asarray(value, dtype=np.float32)
    if pd.isna(value):
        return np.asarray([], dtype=np.float32)
    return np.asarray(ast.literal_eval(str(value)), dtype=np.float32)


def build_class_targets(labels_df):
    labels_df = labels_df.copy().reset_index(drop=True)
    if 'main_label_name' in labels_df.columns:
        y_class = labels_df['main_label_name'].map(MAIN_LABEL_TO_INDEX).to_numpy(dtype=np.int64)
    elif 'main_label_vector' in labels_df.columns:
        main_matrix = np.vstack(labels_df['main_label_vector'].map(parse_vector_column).to_numpy()).astype(np.float32)
        y_class = np.argmax(main_matrix, axis=1).astype(np.int64)
        labels_df['main_label_name'] = [INDEX_TO_MAIN_LABEL[int(i)] for i in y_class]
    else:
        raise ValueError('Dataset must contain main_label_name or main_label_vector for strict 4-class training.')

    if np.any(pd.isna(y_class)):
        raise ValueError('Found labels outside MAIN_LABELS.')
    labels_df['class_index'] = y_class.astype(int)
    return y_class.astype(np.int64), labels_df


def validate_classification_labels(labels_df, y_class, split_name):
    if y_class.ndim != 1:
        raise ValueError(f'{split_name}: class target must be 1D, got {y_class.shape}')
    if not np.isin(y_class, np.arange(len(MAIN_LABELS))).all():
        raise ValueError(f'{split_name}: class target contains invalid class index')
    counts = pd.Series(y_class).value_counts().reindex(range(len(MAIN_LABELS)), fill_value=0)
    counts.index = MAIN_LABELS
    logger.info('%s class distribution: %s', split_name, counts.to_dict())
    print(f'=== {split_name} classification label validation ===')
    display(counts.rename('count').to_frame())
    example_cols = [col for col in ['beat_id', 'record_id', 'raw_label', 'clinical_sub_label', 'main_label_name', 'class_index'] if col in labels_df.columns]
    display(labels_df[example_cols].head(10))


y_train, labels_train = build_class_targets(labels_train)
y_val, labels_val = build_class_targets(labels_val)
y_test, labels_test = build_class_targets(labels_test)

validate_classification_labels(labels_train, y_train, 'train')
validate_classification_labels(labels_val, y_val, 'val')
validate_classification_labels(labels_test, y_test, 'test')


def downsample_train_beats_per_class(x, y, labels_df, beats_per_class, seed=42):
    rng = np.random.default_rng(seed)
    selected_indices = []
    rows = []
    for class_index, class_name in enumerate(MAIN_LABELS):
        class_indices = np.flatnonzero(y == class_index)
        if len(class_indices) == 0:
            raise ValueError(f'No training beats found for class {class_name}.')
        n_select = min(int(beats_per_class), len(class_indices))
        chosen = rng.choice(class_indices, size=n_select, replace=False)
        selected_indices.append(chosen)
        rows.append({
            'class_index': int(class_index),
            'class_name': class_name,
            'available_train_beats': int(len(class_indices)),
            'selected_train_beats': int(n_select),
        })
    selected_indices = np.concatenate(selected_indices)
    rng.shuffle(selected_indices)
    summary = pd.DataFrame(rows)
    return x[selected_indices], y[selected_indices], labels_df.iloc[selected_indices].reset_index(drop=True), summary

if CONFIG.get('balance_train_beats', False):
    x_train, y_train, labels_train, train_balance_summary = downsample_train_beats_per_class(
        x_train,
        y_train,
        labels_train,
        CONFIG['train_beats_per_class'],
        seed=CONFIG['seed'],
    )
    train_balance_summary.to_csv(CONFIG_DIR / 'train_beat_balance_summary.csv', index=False)
    logger.info('Training beats downsampled per class: %s', train_balance_summary.to_dict(orient='records'))
    print('=== balanced training beat subset ===')
    display(train_balance_summary)
    validate_classification_labels(labels_train, y_train, 'train_balanced')
else:
    train_balance_summary = None

logger.info('Classification label mapping completed.')


## 5. Dataset and DataLoader


In [ ]:
logger.info('Dataset normalization and DataLoader setup started.')

class PerLeadZScore:
    def __init__(self, eps=1e-6):
        self.eps = eps
        self.mean_ = None
        self.std_ = None

    def fit(self, x):
        self.mean_ = x.mean(axis=(0, 1), keepdims=True)
        self.std_ = x.std(axis=(0, 1), keepdims=True)
        self.std_ = np.maximum(self.std_, self.eps)
        return self

    def transform(self, x):
        return ((x - self.mean_) / self.std_).astype(np.float32)

normalizer = PerLeadZScore().fit(x_train)
x_train_n = normalizer.transform(x_train)
x_val_n = normalizer.transform(x_val)
x_test_n = normalizer.transform(x_test)
np.save(TRANSFER_DIR / 'normalizer_mean.npy', normalizer.mean_.astype(np.float32))
np.save(TRANSFER_DIR / 'normalizer_std.npy', normalizer.std_.astype(np.float32))


class ECGClassificationDataset(Dataset):
    def __init__(self, x, y, labels_df=None):
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.labels_df = labels_df.reset_index(drop=True) if labels_df is not None else None

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return {'x': self.x[idx], 'y': self.y[idx], 'idx': idx}

train_ds = ECGClassificationDataset(x_train_n, y_train, labels_train)
val_ds = ECGClassificationDataset(x_val_n, y_val, labels_val)
test_ds = ECGClassificationDataset(x_test_n, y_test, labels_test)

train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=CONFIG['num_workers'], pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=torch.cuda.is_available())

logger.info('Dataset normalization and DataLoader setup completed.')
logger.info('DataLoaders ready: train=%d val=%d test=%d', len(train_ds), len(val_ds), len(test_ds))
logger.info('DataLoader batches: train=%d val=%d test=%d', len(train_loader), len(val_loader), len(test_loader))


## 6. LeadPrior-Net classification model definition


In [ ]:
logger.info('LeadPrior-Net classification model definition and instantiation started.')

class SharedLeadCNNStem(nn.Module):
    def __init__(self, in_channels=1, stem_channels=32, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, stem_channels, kernel_size=7, padding=3, bias=False),
            nn.BatchNorm1d(stem_channels),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class MultiScaleResidualCNNBlock(nn.Module):
    def __init__(self, channels, kernels=(3, 7, 15, 31), dropout=0.1):
        super().__init__()
        branch_channels = channels // len(kernels)
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(channels, branch_channels, kernel_size=k, padding=k // 2, bias=False),
                nn.BatchNorm1d(branch_channels),
                nn.GELU(),
            )
            for k in kernels
        ])
        self.project = nn.Sequential(
            nn.Conv1d(branch_channels * len(kernels), channels, kernel_size=1, bias=False),
            nn.BatchNorm1d(channels),
            nn.Dropout(dropout),
        )
        self.act = nn.GELU()

    def forward(self, x):
        out = torch.cat([branch(x) for branch in self.branches], dim=1)
        out = self.project(out)
        return self.act(out + x)


class LeadTokenBuilder(nn.Module):
    def __init__(self, in_channels, token_dim):
        super().__init__()
        self.proj = nn.Linear(in_channels, token_dim)

    def forward(self, lead_features):
        pooled = lead_features.mean(dim=-1)
        return self.proj(pooled)


class LeadPriorModule(nn.Module):
    def __init__(self, token_dim, class_prior_matrix, prior_strength=4.0, prior_learnable=True, dropout=0.1):
        super().__init__()
        prior = torch.tensor(class_prior_matrix, dtype=torch.float32).clamp_min(1e-6)
        prior = prior / prior.sum(dim=-1, keepdim=True)
        prior_bias = torch.log(prior) * float(prior_strength)
        self.class_queries = nn.Parameter(torch.randn(prior.shape[0], token_dim) * 0.02)
        if prior_learnable:
            self.class_lead_prior_bias = nn.Parameter(prior_bias)
        else:
            self.register_buffer('class_lead_prior_bias', prior_bias)
        self.register_buffer('initial_class_prior', prior)
        self.dropout = nn.Dropout(dropout)
        self.scale = token_dim ** -0.5

    def forward(self, lead_tokens):
        data_scores = torch.einsum('bld,qd->bql', lead_tokens, self.class_queries) * self.scale
        scores = data_scores + self.class_lead_prior_bias.unsqueeze(0)
        class_attention = torch.softmax(scores, dim=-1)
        class_context = torch.einsum('bql,bld->bqd', class_attention, lead_tokens)
        class_context = self.dropout(class_context)
        lead_attention = class_attention.mean(dim=1)
        lead_attention = lead_attention / lead_attention.sum(dim=-1, keepdim=True).clamp_min(1e-6)
        return class_context, class_attention, lead_attention


class LeadTimeAttentionPooling(nn.Module):
    def __init__(self, token_dim, dropout=0.1):
        super().__init__()
        self.score = nn.Sequential(
            nn.LayerNorm(token_dim),
            nn.Linear(token_dim, token_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(token_dim // 2, 1),
        )

    def forward(self, tokens):
        scores = self.score(tokens).squeeze(-1)
        weights = torch.softmax(scores, dim=1)
        pooled = torch.einsum('bn,bnd->bd', weights, tokens)
        return pooled, weights


class LeadPriorNetBackbone(nn.Module):
    def __init__(self, config):
        super().__init__()
        m = config['model']
        self.input_leads = m['input_leads']
        self.temporal_segments = m['temporal_segments']
        self.stem = SharedLeadCNNStem(1, m['stem_channels'], m['dropout'])
        self.channel_project = nn.Sequential(
            nn.Conv1d(m['stem_channels'], m['cnn_channels'], kernel_size=1, bias=False),
            nn.BatchNorm1d(m['cnn_channels']),
            nn.GELU(),
        )
        self.multi_scale = MultiScaleResidualCNNBlock(m['cnn_channels'], dropout=m['dropout'])
        self.token_builder = LeadTokenBuilder(m['cnn_channels'], m['token_dim'])
        self.lead_prior = LeadPriorModule(m['token_dim'], MAIN_CLASS_PRIOR_MATRIX, prior_strength=config['attention_prior_strength'], prior_learnable=config['attention_prior_learnable'], dropout=m['dropout'])
        cross_layer = nn.TransformerEncoderLayer(
            d_model=m['token_dim'], nhead=m['num_heads'], dim_feedforward=m['transformer_ff_dim'],
            dropout=m['dropout'], batch_first=True, activation='gelu', norm_first=True,
        )
        self.cross_lead_transformer = nn.TransformerEncoder(cross_layer, num_layers=m['cross_lead_layers'])
        temporal_layer = nn.TransformerEncoderLayer(
            d_model=m['token_dim'], nhead=m['num_heads'], dim_feedforward=m['transformer_ff_dim'],
            dropout=m['dropout'], batch_first=True, activation='gelu', norm_first=True,
        )
        self.temporal_transformer = nn.TransformerEncoder(temporal_layer, num_layers=m['temporal_layers'])
        self.pooling = LeadTimeAttentionPooling(m['token_dim'], dropout=m['dropout'])
        self.norm = nn.LayerNorm(m['token_dim'] * 2)

    def forward(self, x):
        b, t, l = x.shape
        x_lead = x.permute(0, 2, 1).reshape(b * l, 1, t)
        features = self.stem(x_lead)
        features = self.channel_project(features)
        features = self.multi_scale(features)
        lead_features = features.reshape(b, l, features.shape[1], features.shape[2])
        lead_tokens_raw = self.token_builder(lead_features)
        class_context, class_attention, lead_attention = self.lead_prior(lead_tokens_raw)
        lead_tokens = self.cross_lead_transformer(lead_tokens_raw)

        segment_features = F.adaptive_avg_pool1d(features, self.temporal_segments)
        segment_features = segment_features.reshape(b, l, segment_features.shape[1], self.temporal_segments).mean(dim=1)
        temporal_tokens = segment_features.permute(0, 2, 1)
        temporal_tokens = self.temporal_transformer(self.token_builder.proj(temporal_tokens))

        lead_pooled, lead_pool_weights = self.pooling(lead_tokens)
        time_pooled, time_pool_weights = self.pooling(temporal_tokens)
        pooled_features = self.norm(torch.cat([lead_pooled, time_pooled], dim=-1))
        return {
            'pooled_features': pooled_features,
            'lead_tokens': lead_tokens,
            'temporal_tokens': temporal_tokens,
            'class_context': class_context,
            'class_attention': class_attention,
            'lead_attention': lead_attention,
            'lead_pool_weights': lead_pool_weights,
            'time_pool_weights': time_pool_weights,
        }


class LeadPriorNet(nn.Module):
    def __init__(self, config):
        super().__init__()
        m = config['model']
        self.backbone = LeadPriorNetBackbone(config)
        self.classification_head = nn.Sequential(
            nn.LayerNorm(m['token_dim'] * 2),
            nn.Linear(m['token_dim'] * 2, m['token_dim']),
            nn.GELU(),
            nn.Dropout(m['dropout']),
            nn.Linear(m['token_dim'], m['main_outputs']),
        )

    def forward(self, x):
        backbone_out = self.backbone(x)
        logits = self.classification_head(backbone_out['pooled_features'])
        return {
            'logits': logits,
            'probabilities': torch.softmax(logits, dim=1),
            **backbone_out,
        }


def freeze_for_transfer(model, scheme):
    for param in model.parameters():
        param.requires_grad = True
    if scheme == 'frozen_backbone':
        for param in model.backbone.parameters():
            param.requires_grad = False
    elif scheme == 'partial_finetune':
        for module in [model.backbone.stem, model.backbone.channel_project, model.backbone.multi_scale]:
            for param in module.parameters():
                param.requires_grad = False
    elif scheme in {'full_finetune', 'no_pretrain'}:
        pass
    else:
        raise ValueError(f'Unknown transfer scheme: {scheme}')


def backbone_state_dict(model):
    return model.backbone.state_dict()


def heads_state_dict(model):
    return {'classification_head': model.classification_head.state_dict()}

model = LeadPriorNet(CONFIG).to(CONFIG['device'])
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
logger.info('Model instantiated. Total params=%d | Trainable params=%d', total_params, trainable_params)
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
logger.info('LeadPrior-Net classification model definition and instantiation completed.')


## 7. Classification loss functions


In [ ]:
logger.info('Loss, optimizer, scheduler, and one-batch sanity check setup started.')

criterion = nn.CrossEntropyLoss().to(CONFIG['device'])
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=CONFIG['scheduler_patience'])

sanity_batch = next(iter(train_loader))
model.train()
optimizer.zero_grad(set_to_none=True)
sanity_outputs = model(sanity_batch['x'].to(CONFIG['device']))
batch_size = sanity_batch['x'].shape[0]
required_shapes = {
    'logits': (batch_size, len(MAIN_LABELS)),
    'probabilities': (batch_size, len(MAIN_LABELS)),
    'class_attention': (batch_size, len(MAIN_LABELS), len(LEAD_ORDER)),
    'class_context': (batch_size, len(MAIN_LABELS), CONFIG['model']['token_dim']),
    'lead_attention': (batch_size, len(LEAD_ORDER)),
}
for key, shape in required_shapes.items():
    if tuple(sanity_outputs[key].shape) != shape:
        raise ValueError(f'{key} shape mismatch. Expected {shape}, got {tuple(sanity_outputs[key].shape)}')
sanity_loss = criterion(sanity_outputs['logits'], sanity_batch['y'].to(CONFIG['device']))
sanity_loss.backward()
optimizer.zero_grad(set_to_none=True)
logger.info('One-batch classification sanity check passed. CE loss=%.6f', float(sanity_loss.detach().cpu()))
logger.info('Loss, optimizer, scheduler, and one-batch sanity check setup completed.')


## 8. Metrics


In [ ]:
def softmax_np(logits):
    logits = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)


def compute_classification_metrics(y_true, logits, label_names):
    prob = softmax_np(logits)
    pred = np.argmax(prob, axis=1)
    per_class_f1 = f1_score(y_true, pred, labels=np.arange(len(label_names)), average=None, zero_division=0)
    cm = confusion_matrix(y_true, pred, labels=np.arange(len(label_names)))
    metrics = {
        'accuracy': float((pred == y_true).mean()),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, pred)),
        'macro_f1': float(f1_score(y_true, pred, average='macro', zero_division=0)),
        'micro_f1': float(f1_score(y_true, pred, average='micro', zero_division=0)),
        'weighted_f1': float(f1_score(y_true, pred, average='weighted', zero_division=0)),
    }
    class_rows = []
    auc_values = []
    for idx, (label, display_label, f1_value) in enumerate(zip(label_names, MAIN_LABEL_DISPLAY, per_class_f1)):
        tp = float(cm[idx, idx])
        fn = float(cm[idx, :].sum() - cm[idx, idx])
        fp = float(cm[:, idx].sum() - cm[idx, idx])
        tn = float(cm.sum() - tp - fn - fp)
        sensitivity = tp / max(tp + fn, 1.0)
        specificity = tn / max(tn + fp, 1.0)
        try:
            auc_value = float(roc_auc_score((y_true == idx).astype(int), prob[:, idx]))
            auc_values.append(auc_value)
        except ValueError:
            auc_value = np.nan
        metrics[f'f1_{label}'] = float(f1_value)
        metrics[f'sensitivity_{label}'] = float(sensitivity)
        metrics[f'specificity_{label}'] = float(specificity)
        metrics[f'auc_{label}'] = float(auc_value) if np.isfinite(auc_value) else np.nan
        class_rows.append({
            'label': display_label,
            'internal_label': label,
            'f1_score': float(f1_value),
            'sensitivity': float(sensitivity),
            'specificity': float(specificity),
            'auc': float(auc_value) if np.isfinite(auc_value) else np.nan,
            'support': int((y_true == idx).sum()),
        })
    metrics['macro_auc'] = float(np.nanmean(auc_values)) if auc_values else np.nan
    return metrics, prob, pred, pd.DataFrame(class_rows)

def plot_training_curves(metrics_df):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=300)
    axes[0].plot(metrics_df['epoch'], metrics_df['train_loss'], label='train')
    axes[0].plot(metrics_df['epoch'], metrics_df['val_loss'], label='val')
    axes[0].set_title('Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[1].plot(metrics_df['epoch'], metrics_df['train_macro_f1'], label='train')
    axes[1].plot(metrics_df['epoch'], metrics_df['val_macro_f1'], label='val')
    axes[1].set_title('Macro F1')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(PLOT_DIR / 'training_curve.png', bbox_inches='tight')
    fig.savefig(METRICS_DIR / 'training_curve.png', bbox_inches='tight')
    plt.close(fig)


def save_confusion_matrix(y_true, y_pred, labels, output_prefix):
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(labels)))
    pd.DataFrame(cm, index=labels, columns=labels).to_csv(METRICS_DIR / f'{output_prefix}.csv')
    fig, ax = plt.subplots(figsize=(9, 8), dpi=400)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', xticklabels=labels, yticklabels=labels, ax=ax, cbar=True, linewidths=1.0, linecolor='black', annot_kws={'fontsize': 14})
    ax.set_xlabel('Predicted', fontsize=14)
    ax.set_ylabel('True', fontsize=14)
    ax.set_title(output_prefix.replace('_', ' ').title(), fontsize=16, weight='bold')
    fig.tight_layout()
    fig.savefig(METRICS_DIR / f'{output_prefix}.png', bbox_inches='tight')
    plt.close(fig)
    return cm


def save_roc_curves(y_true, prob, output_prefix):
    rows = []
    fig, ax = plt.subplots(figsize=(9, 7), dpi=400)
    for idx, label in enumerate(MAIN_LABEL_DISPLAY):
        binary_true = (y_true == idx).astype(int)
        if binary_true.min() == binary_true.max():
            continue
        fpr, tpr, _ = roc_curve(binary_true, prob[:, idx])
        auc_value = roc_auc_score(binary_true, prob[:, idx])
        ax.plot(fpr, tpr, linewidth=2.2, label=f'{label} AUC={auc_value:.3f}')
        for x, y in zip(fpr, tpr):
            rows.append({'label': label, 'fpr': float(x), 'tpr': float(y), 'auc': float(auc_value)})
    ax.plot([0, 1], [0, 1], linestyle='--', color='black', linewidth=1.2)
    ax.set_xlabel('False Positive Rate', fontsize=13)
    ax.set_ylabel('True Positive Rate / Sensitivity', fontsize=13)
    ax.set_title(output_prefix.replace('_', ' ').title(), fontsize=16, weight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f'{output_prefix}.png', bbox_inches='tight')
    fig.savefig(METRICS_DIR / f'{output_prefix}.png', bbox_inches='tight')
    plt.close(fig)
    pd.DataFrame(rows).to_csv(METRICS_DIR / f'{output_prefix}.csv', index=False)


def save_pr_curves(y_true, prob, output_prefix):
    rows = []
    fig, ax = plt.subplots(figsize=(9, 7), dpi=400)
    for idx, label in enumerate(MAIN_LABEL_DISPLAY):
        binary_true = (y_true == idx).astype(int)
        if binary_true.min() == binary_true.max():
            continue
        precision, recall, _ = precision_recall_curve(binary_true, prob[:, idx])
        auprc_value = average_precision_score(binary_true, prob[:, idx])
        ax.plot(recall, precision, linewidth=2.2, label=f'{label} AUPRC={auprc_value:.3f}')
        for r, p in zip(recall, precision):
            rows.append({'label': label, 'recall': float(r), 'precision': float(p), 'auprc': float(auprc_value)})
    ax.set_xlabel('Recall / Sensitivity', fontsize=13)
    ax.set_ylabel('Precision', fontsize=13)
    ax.set_title(output_prefix.replace('_', ' ').title(), fontsize=16, weight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f'{output_prefix}.png', bbox_inches='tight')
    fig.savefig(METRICS_DIR / f'{output_prefix}.png', bbox_inches='tight')
    plt.close(fig)
    pd.DataFrame(rows).to_csv(METRICS_DIR / f'{output_prefix}.csv', index=False)


## 9. Training loop


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, device='cpu'):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    n = 0
    logits_all, y_all, attn_all, class_attn_all, idx_all = [], [], [], [], []
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for batch in loader:
            x = batch['x'].to(device)
            y = batch['y'].to(device)
            if training:
                optimizer.zero_grad(set_to_none=True)
            outputs = model(x)
            loss = criterion(outputs['logits'], y)
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            batch_size = x.size(0)
            total_loss += float(loss.detach().cpu()) * batch_size
            n += batch_size
            logits_all.append(outputs['logits'].detach().cpu().numpy())
            y_all.append(y.detach().cpu().numpy())
            attn_all.append(outputs['lead_attention'].detach().cpu().numpy())
            class_attn_all.append(outputs['class_attention'].detach().cpu().numpy())
            idx_all.append(batch['idx'].detach().cpu().numpy())
    return {
        'loss': total_loss / max(n, 1),
        'logits': np.concatenate(logits_all),
        'y_true': np.concatenate(y_all),
        'lead_attention': np.concatenate(attn_all),
        'class_attention': np.concatenate(class_attn_all),
        'idx': np.concatenate(idx_all),
    }


def save_checkpoint(path, epoch, metric, val_loss):
    torch.save({
        'model_state_dict': model.state_dict(),
        'backbone_state_dict': backbone_state_dict(model),
        'heads_state_dict': heads_state_dict(model),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'epoch': epoch,
        'best_metric': metric,
        'best_val_macro_f1': metric,
        'val_loss': val_loss,
        'config': CONFIG,
        'label_mapping': {'main_labels': MAIN_LABELS, 'main_label_display': MAIN_LABEL_DISPLAY},
        'lead_order': LEAD_ORDER,
        'main_label_names': MAIN_LABELS,
        'main_class_prior_matrix': MAIN_CLASS_PRIOR_MATRIX.tolist(),
        'main_class_lead_priors': MAIN_CLASS_LEAD_PRIORS,
        'task_type': 'single_label_4_class_softmax',
    }, path)

metrics_rows = []
best_macro_f1 = -np.inf
best_val_loss = np.inf
best_epoch = 0
patience_counter = 0

try:
    logger.info('Training started for %d epochs.', CONFIG['epochs'])
    for epoch in range(1, CONFIG['epochs'] + 1):
        epoch_start = time.time()
        train_state = run_epoch(model, train_loader, criterion, optimizer=optimizer, device=CONFIG['device'])
        val_state = run_epoch(model, val_loader, criterion, optimizer=None, device=CONFIG['device'])
        train_metrics, _, _, _ = compute_classification_metrics(train_state['y_true'], train_state['logits'], MAIN_LABELS)
        val_metrics, _, _, _ = compute_classification_metrics(val_state['y_true'], val_state['logits'], MAIN_LABELS)
        scheduler.step(val_metrics['macro_f1'])
        lr = optimizer.param_groups[0]['lr']
        row = {
            'epoch': epoch,
            'train_loss': train_state['loss'],
            'val_loss': val_state['loss'],
            'train_macro_f1': train_metrics['macro_f1'],
            'val_macro_f1': val_metrics['macro_f1'],
            'train_accuracy': train_metrics['accuracy'],
            'val_accuracy': val_metrics['accuracy'],
            'learning_rate': lr,
            'epoch_time': time.time() - epoch_start,
        }
        for label in MAIN_LABELS:
            row[f'train_f1_{label}'] = train_metrics[f'f1_{label}']
            row[f'val_f1_{label}'] = val_metrics[f'f1_{label}']
        metrics_rows.append(row)
        pd.DataFrame(metrics_rows).to_csv(LOG_DIR / 'metrics.csv', index=False)
        pd.DataFrame(metrics_rows).to_csv(METRICS_DIR / 'metrics.csv', index=False)
        logger.info(
            'Epoch %03d/%03d | train_loss=%.5f val_loss=%.5f train_macro_f1=%.4f val_macro_f1=%.4f lr=%.3e time=%.1fs',
            epoch, CONFIG['epochs'], train_state['loss'], val_state['loss'], train_metrics['macro_f1'], val_metrics['macro_f1'], lr, row['epoch_time']
        )
        save_checkpoint(CKPT_DIR / 'last.pt', epoch, best_macro_f1, val_state['loss'])
        if val_metrics['macro_f1'] > best_macro_f1:
            best_macro_f1 = val_metrics['macro_f1']
            best_epoch = epoch
            patience_counter = 0
            save_checkpoint(CKPT_DIR / 'best_macro_f1.pt', epoch, best_macro_f1, val_state['loss'])
            logger.info('New best val macro F1 %.4f at epoch %d', best_macro_f1, epoch)
        else:
            patience_counter += 1
        if val_state['loss'] < best_val_loss:
            best_val_loss = val_state['loss']
            save_checkpoint(CKPT_DIR / 'best_val_loss.pt', epoch, best_macro_f1, val_state['loss'])
        if epoch >= 5 and epoch % 5 == 0:
            periodic_path = PER_5_EPOCH_CKPT_DIR / f'epoch_{epoch:03d}.pt'
            save_checkpoint(periodic_path, epoch, best_macro_f1, val_state['loss'])
            logger.info('Saved periodic 5-epoch checkpoint: %s', periodic_path)
        if CONFIG.get('early_stopping_enabled', False) and patience_counter >= CONFIG['early_stopping_patience']:
            logger.info('Early stopping triggered at epoch %d', epoch)
            break
    logger.info('Training finished. Best epoch=%d | best_val_macro_f1=%.4f', best_epoch, best_macro_f1)
except Exception:
    logger.exception('Training failed.')
    try:
        save_checkpoint(CKPT_DIR / 'last.pt', len(metrics_rows), best_macro_f1, np.nan)
    except Exception:
        logger.exception('Failed to save last checkpoint after error.')
    raise


## 10. Validation and testing


In [ ]:
logger.info('Final validation and test evaluation started.')
best_ckpt_path = CKPT_DIR / 'best_macro_f1.pt'
if best_ckpt_path.exists():
    ckpt = torch.load(best_ckpt_path, map_location=CONFIG['device'])
    model.load_state_dict(ckpt['model_state_dict'])
    logger.info('Loaded best macro-F1 checkpoint from epoch %s for final validation/test evaluation', ckpt.get('epoch'))

val_state = run_epoch(model, val_loader, criterion, optimizer=None, device=CONFIG['device'])
test_state = run_epoch(model, test_loader, criterion, optimizer=None, device=CONFIG['device'])
val_metrics, val_prob, val_pred, val_class_metrics = compute_classification_metrics(val_state['y_true'], val_state['logits'], MAIN_LABELS)
test_metrics, test_prob, test_pred, test_class_metrics = compute_classification_metrics(test_state['y_true'], test_state['logits'], MAIN_LABELS)
logger.info('Final validation metrics: %s', val_metrics)
logger.info('Final test metrics: %s', test_metrics)
display(pd.DataFrame([val_metrics, test_metrics], index=['val', 'test']))


def save_predictions(path, labels_df, y_true, prob, pred):
    df = labels_df.copy().reset_index(drop=True)
    df['true_class_index'] = y_true.astype(int)
    df['pred_class_index'] = pred.astype(int)
    df['true_label'] = [MAIN_LABELS[int(i)] for i in y_true]
    df['pred_label'] = [MAIN_LABELS[int(i)] for i in pred]
    for i, name in enumerate(MAIN_LABELS):
        df[f'prob_{name}'] = prob[:, i]
    df.to_csv(path, index=False)
    return df

val_predictions = save_predictions(PRED_DIR / 'val_predictions.csv', labels_val.iloc[val_state['idx']], val_state['y_true'], val_prob, val_pred)
test_predictions = save_predictions(PRED_DIR / 'test_predictions.csv', labels_test.iloc[test_state['idx']], test_state['y_true'], test_prob, test_pred)

pd.DataFrame([
    {'split': 'val', **val_metrics, 'loss': val_state['loss']},
    {'split': 'test', **test_metrics, 'loss': test_state['loss']},
]).to_csv(LOG_DIR / 'final_metrics.csv', index=False)
pd.DataFrame([
    {'split': 'val', **val_metrics, 'loss': val_state['loss']},
    {'split': 'test', **test_metrics, 'loss': test_state['loss']},
]).to_csv(METRICS_DIR / 'final_metrics.csv', index=False)

save_confusion_matrix(val_state['y_true'], val_pred, MAIN_LABEL_DISPLAY, 'val_confusion_matrix')
save_confusion_matrix(test_state['y_true'], test_pred, MAIN_LABEL_DISPLAY, 'test_confusion_matrix')

test_f1_rows = []
for label, display_label in zip(MAIN_LABELS, MAIN_LABEL_DISPLAY):
    test_f1_rows.append({
        'split': 'test',
        'label': display_label,
        'internal_label': label,
        'f1_score': test_metrics[f'f1_{label}'],
    })
test_f1_df = pd.DataFrame(test_f1_rows)
test_f1_df.to_csv(PLOT_DIR / 'test_f1_scores_by_class.csv', index=False)
test_f1_df.to_csv(METRICS_DIR / 'test_f1_scores_by_class.csv', index=False)
val_class_metrics.insert(0, 'split', 'val')
test_class_metrics.insert(0, 'split', 'test')
pd.concat([val_class_metrics, test_class_metrics], ignore_index=True).to_csv(METRICS_DIR / 'per_class_metrics.csv', index=False)
save_roc_curves(val_state['y_true'], val_prob, 'val_roc_curve')
save_roc_curves(test_state['y_true'], test_prob, 'test_roc_curve')
save_pr_curves(val_state['y_true'], val_prob, 'val_pr_curve')
save_pr_curves(test_state['y_true'], test_prob, 'test_pr_curve')
pd.DataFrame([
    {'split': 'val', 'macro_f1': val_metrics['macro_f1'], 'weighted_f1': val_metrics['weighted_f1'], 'balanced_accuracy': val_metrics['balanced_accuracy'], 'macro_auc': val_metrics['macro_auc'], 'accuracy': val_metrics['accuracy']},
    {'split': 'test', 'macro_f1': test_metrics['macro_f1'], 'weighted_f1': test_metrics['weighted_f1'], 'balanced_accuracy': test_metrics['balanced_accuracy'], 'macro_auc': test_metrics['macro_auc'], 'accuracy': test_metrics['accuracy']},
]).to_csv(PLOT_DIR / 'f1_score_summary.csv', index=False)
pd.DataFrame([
    {'split': 'val', 'macro_f1': val_metrics['macro_f1'], 'weighted_f1': val_metrics['weighted_f1'], 'balanced_accuracy': val_metrics['balanced_accuracy'], 'macro_auc': val_metrics['macro_auc'], 'accuracy': val_metrics['accuracy']},
    {'split': 'test', 'macro_f1': test_metrics['macro_f1'], 'weighted_f1': test_metrics['weighted_f1'], 'balanced_accuracy': test_metrics['balanced_accuracy'], 'macro_auc': test_metrics['macro_auc'], 'accuracy': test_metrics['accuracy']},
]).to_csv(METRICS_DIR / 'f1_score_summary.csv', index=False)
logger.info('Final validation and test evaluation completed.')


## 11. Save transfer-ready artifacts


In [ ]:
logger.info('Transfer-ready artifact export started.')

def save_transfer_ready_artifacts():
    label_mapping = {
        'task_type': 'single_label_4_class_softmax',
        'main_labels': MAIN_LABELS,
        'main_label_display': MAIN_LABEL_DISPLAY,
        'class_to_index': MAIN_LABEL_TO_INDEX,
        'index_to_class': INDEX_TO_MAIN_LABEL,
    }
    backbone_payload = {
        'backbone_state_dict': model.backbone.state_dict(),
        'config': CONFIG,
        'label_mapping': label_mapping,
        'lead_order': LEAD_ORDER,
        'main_class_prior_matrix': MAIN_CLASS_PRIOR_MATRIX.tolist(),
        'main_class_lead_priors': MAIN_CLASS_LEAD_PRIORS,
        'normalizer_mean_path': 'normalizer_mean.npy',
        'normalizer_std_path': 'normalizer_std.npy',
        'compatible_transfer_schemes': ['frozen_backbone', 'partial_finetune', 'full_finetune'],
    }
    full_payload = {
        'model_state_dict': model.state_dict(),
        'backbone_state_dict': model.backbone.state_dict(),
        'heads_state_dict': heads_state_dict(model),
        'config': CONFIG,
        'label_mapping': label_mapping,
        'lead_order': LEAD_ORDER,
        'main_label_names': MAIN_LABELS,
        'main_class_prior_matrix': MAIN_CLASS_PRIOR_MATRIX.tolist(),
        'main_class_lead_priors': MAIN_CLASS_LEAD_PRIORS,
    }
    torch.save(backbone_payload, TRANSFER_DIR / 'leadprior_backbone.pt')
    torch.save(full_payload, TRANSFER_DIR / 'leadprior_full_model.pt')
    torch.save({'heads_state_dict': heads_state_dict(model), 'config': CONFIG, 'label_mapping': label_mapping}, TRANSFER_DIR / 'leadprior_heads.pt')
    with open(TRANSFER_DIR / 'label_mapping.json', 'w') as f:
        json.dump(label_mapping, f, indent=2)
    with open(TRANSFER_DIR / 'model_config.json', 'w') as f:
        json.dump(CONFIG['model'], f, indent=2)
    with open(TRANSFER_DIR / 'lead_prior_config.json', 'w') as f:
        json.dump(LEAD_PRIOR_CONFIG, f, indent=2)
    logger.info('Transfer-ready artifacts saved to %s', TRANSFER_DIR)

save_transfer_ready_artifacts()
logger.info('Transfer-ready artifact export completed.')


## 12. Plot diagnostics


In [ ]:
logger.info('Plot and diagnostic artifact generation started.')
metrics_path = LOG_DIR / 'metrics.csv'
if metrics_path.exists():
    metrics_df = pd.read_csv(metrics_path)
    if not metrics_df.empty:
        plot_training_curves(metrics_df)

pd.DataFrame(MAIN_CLASS_PRIOR_MATRIX, index=MAIN_LABEL_DISPLAY, columns=LEAD_ORDER).to_csv(PLOT_DIR / 'initial_attention_prior_matrix.csv')
pd.DataFrame(MAIN_CLASS_PRIOR_MATRIX, index=MAIN_LABEL_DISPLAY, columns=LEAD_ORDER).to_csv(METRICS_DIR / 'initial_attention_prior_matrix.csv')
mean_class_attention = test_state['class_attention'].mean(axis=0)
pd.DataFrame(mean_class_attention, index=MAIN_LABEL_DISPLAY, columns=LEAD_ORDER).to_csv(PLOT_DIR / 'lead_attention_heatmap.csv')
pd.DataFrame(mean_class_attention, index=MAIN_LABEL_DISPLAY, columns=LEAD_ORDER).to_csv(METRICS_DIR / 'lead_attention_heatmap.csv')
fig, ax = plt.subplots(figsize=(12, 5), dpi=400)
sns.heatmap(mean_class_attention, cmap='Oranges', xticklabels=LEAD_ORDER, yticklabels=MAIN_LABEL_DISPLAY, ax=ax, linewidths=1.0, linecolor='black')
ax.set_title('Mean Class-Specific Lead Attention', fontsize=16, weight='bold')
ax.set_xlabel('Lead')
ax.set_ylabel('Class Query')
fig.tight_layout()
fig.savefig(PLOT_DIR / 'lead_attention_heatmap.png', bbox_inches='tight')
fig.savefig(METRICS_DIR / 'lead_attention_heatmap.png', bbox_inches='tight')
plt.close(fig)
logger.info('Plot and diagnostic artifact generation completed.')


## 13. Sanity check for transfer learning compatibility


In [ ]:
logger.info('Transfer compatibility sanity check started.')
backbone_payload = torch.load(TRANSFER_DIR / 'leadprior_backbone.pt', map_location='cpu')
full_payload = torch.load(TRANSFER_DIR / 'leadprior_full_model.pt', map_location='cpu')
heads_payload = torch.load(TRANSFER_DIR / 'leadprior_heads.pt', map_location='cpu')

scratch_model = LeadPriorNet(CONFIG)
frozen_model = LeadPriorNet(CONFIG)
frozen_model.backbone.load_state_dict(backbone_payload['backbone_state_dict'])
freeze_for_transfer(frozen_model, 'frozen_backbone')
partial_model = LeadPriorNet(CONFIG)
partial_model.backbone.load_state_dict(backbone_payload['backbone_state_dict'])
freeze_for_transfer(partial_model, 'partial_finetune')
full_model = LeadPriorNet(CONFIG)
full_model.load_state_dict(full_payload['model_state_dict'])
freeze_for_transfer(full_model, 'full_finetune')

expected_head_keys = {'classification_head'}
actual_head_keys = set(heads_payload['heads_state_dict'].keys())
if actual_head_keys != expected_head_keys:
    raise ValueError(f'Head artifact is incomplete. Expected {sorted(expected_head_keys)}, got {sorted(actual_head_keys)}')

sanity = pd.DataFrame([
    {'scheme': 'no_pretrain', 'trainable_params': sum(p.numel() for p in scratch_model.parameters() if p.requires_grad)},
    {'scheme': 'frozen_backbone', 'trainable_params': sum(p.numel() for p in frozen_model.parameters() if p.requires_grad)},
    {'scheme': 'partial_finetune', 'trainable_params': sum(p.numel() for p in partial_model.parameters() if p.requires_grad)},
    {'scheme': 'full_finetune', 'trainable_params': sum(p.numel() for p in full_model.parameters() if p.requires_grad)},
])
display(sanity)
logger.info('Transfer compatibility sanity check passed.')
RUN_FINISHED_AT = datetime.now()
RUN_DURATION_SECONDS = time.time() - RUN_START_TIME
logger.info('Run finished at: %s', RUN_FINISHED_AT.strftime('%Y-%m-%d %H:%M:%S'))
logger.info('Total run duration: %.2f seconds (%.2f minutes)', RUN_DURATION_SECONDS, RUN_DURATION_SECONDS / 60.0)
logger.info('LeadPrior-Net classification-only pretraining notebook completed successfully. Output directory: %s', OUTPUT_DIR)
print('Notebook complete. Output:', OUTPUT_DIR)
